# Week 03. REST 응답, DataFrame 변환, UI 함수 설계

3주차는 REST API 응답을 표로 바꾸고, 사용자가 필터 조건을 넣으면 결과를 돌려주는 흐름을 다룹니다.
실제 웹 요청과 API 키 없이 실행되도록 로컬 JSON 데이터를 사용합니다.


## 제출자 정보

- 이름: 이성민
- 학과: 소프트웨어융합과
- 학년: 2학년
- 학번: 2151050


## 1. REST API 응답 구조 흉내 내기

REST API는 대개 상태 코드, 메타데이터, 실제 데이터 목록을 함께 반환합니다.
필요한 것은 `data`에 들어 있는 목록입니다.


In [1]:
import pandas as pd

response = {
    "status_code": 200,
    "data": [
        {"station": "City Hall", "month": "2026-03", "rides": 1240, "weather": "clear"},
        {"station": "City Hall", "month": "2026-04", "rides": 1385, "weather": "cloudy"},
        {"station": "River Park", "month": "2026-03", "rides": 980, "weather": "clear"},
        {"station": "River Park", "month": "2026-04", "rides": 1110, "weather": "rain"},
        {"station": "Campus", "month": "2026-03", "rides": 760, "weather": "cloudy"},
    ],
}

bike_df = pd.DataFrame(response["data"])
bike_df


,station,month,rides,weather
0,City Hall,2026-03,1240,clear
1,City Hall,2026-04,1385,cloudy
2,River Park,2026-03,980,clear
3,River Park,2026-04,1110,rain
4,Campus,2026-03,760,cloudy


## 2. 필터 함수 만들기

Gradio 같은 UI 라이브러리는 결국 Python 함수를 화면에 연결합니다.
먼저 화면 없이도 잘 동작하는 순수 함수를 만들면 테스트가 쉬워집니다.


In [2]:
def filter_rides(station="all", min_rides=0):
    filtered = bike_df.copy()
    if station != "all":
        filtered = filtered[filtered["station"] == station]
    filtered = filtered[filtered["rides"] >= min_rides]
    return filtered.sort_values(["station", "month"]).reset_index(drop=True)


filter_rides(station="City Hall", min_rides=1300)


,station,month,rides,weather
0,City Hall,2026-04,1385,cloudy


## 3. 결과 요약 함수 만들기

사용자가 필터링한 결과를 바로 읽을 수 있도록 요약 문장을 함께 반환합니다.


In [3]:
def ride_dashboard(station="all", min_rides=0):
    result = filter_rides(station, min_rides)
    total = int(result["rides"].sum())
    return {
        "rows": len(result),
        "total_rides": total,
        "preview": result.to_dict(orient="records"),
    }


ride_dashboard("all", 1000)


{'rows': 3,
 'total_rides': 3735,
 'preview': [{'station': 'City Hall',
   'month': '2026-03',
   'rides': 1240,
   'weather': 'clear'},
  {'station': 'City Hall',
   'month': '2026-04',
   'rides': 1385,
   'weather': 'cloudy'},
  {'station': 'River Park',
   'month': '2026-04',
   'rides': 1110,
   'weather': 'rain'}]}

## 정리

- REST 응답의 핵심 데이터는 DataFrame으로 변환하면 분석하기 쉽습니다.
- UI를 붙이기 전 순수 함수로 필터링 로직을 먼저 작성합니다.
- API 키나 외부 서버가 없어도 같은 데이터 처리 개념을 검증할 수 있습니다.
